# IndyCar metrics generator

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import numpy as np
import pandas as pd

from types import TracebackType
from pathlib import Path

from numpy.polynomial.polynomial import polyfit

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/indycar/datasets")
EXTRA_DIR = Path("/content/drive/MyDrive/indycar/scraped_data")
INPUT_FILE = "IndyCar_dataset_v25.csv"   # the results file you typed the latest race into
OUTPUT_FILE = "IndyCar_dataset_v26.csv"

In [ ]:
def computeELO(elos, ranks, k):
  n = len(elos)
  if n <=1:
    return elos

  scores = (n - ranks) / (n-1)

  diff = (elos.reshape(-1,1) - elos.reshape(1,-1)) /400
  expected = 1/(1+10**(-diff))
  np.fill_diagonal(expected, np.nan)
  expected = np.nanmean(expected, axis=1)

  return elos + k * (scores - expected)

In [ ]:
df = pd.read_csv(BASE_DIR / INPUT_FILE)
pit_stops_df = pd.read_csv(EXTRA_DIR / "pit_stops.csv")
race_metadata_df = pd.read_csv(EXTRA_DIR / "race_metadata.csv")
cautions_df = pd.read_csv(EXTRA_DIR / "cautions.csv")
lap_positions_df = pd.read_csv(EXTRA_DIR / "lap_positions.csv")
lap_times_df = pd.read_csv(EXTRA_DIR / "lap_times.csv")

In [ ]:
# Races left out until they're fixed in the results file. Road America 2025 was typed
# with The Thermal Club's EventID (6463), and its real EventID isn't confirmed yet.
EXCLUDED_RACES = [(6463, "Road America")]
for event_id, track in EXCLUDED_RACES:
    df = df[~((df["EventID"] == event_id) & (df["Track"] == track))]

def check_results(df):
    """Stop before building features if the typed-in results look wrong."""
    problems = []
    races = df.groupby("EventID")
    for col in ["EventDate", "TrackID", "FieldSize"]:
        counts = races[col].nunique()
        for event_id in counts[counts > 1].index:
            values = sorted(df.loc[df["EventID"] == event_id, col].astype(str).unique())
            problems.append(f"EventID {event_id} has more than one {col}: {values}")
    for event_id, race in races:
        n = len(race)
        if race["DriverID"].duplicated().any():
            problems.append(f"EventID {event_id} lists the same DriverID twice")
        if sorted(race["PositionFinish"]) != list(range(1, n + 1)):
            problems.append(f"EventID {event_id}: finishing positions aren't 1 to {n}")
        if race["FieldSize"].iloc[0] != n:
            problems.append(f"EventID {event_id}: FieldSize is {race['FieldSize'].iloc[0]} but {n} drivers are listed")
    names = df.groupby("DriverID")["DriverName"].unique()
    for driver_id, driver_names in names[names.map(len) > 1].items():
        problems.append(f"DriverID {driver_id} is used for {', '.join(driver_names)}")
    ids = df.groupby("DriverName")["DriverID"].unique()
    for name, driver_ids in ids[ids.map(len) > 1].items():
        problems.append(f"{name} has more than one DriverID: {', '.join(map(str, driver_ids))}")
    if problems:
        raise ValueError("Fix these in the results file first:\n- " + "\n- ".join(problems))

# Known typing mistakes in the results file, corrected here so the output comes out right.
# Once the file itself is fixed these lines change nothing, so they're safe to keep.
indy500_2014 = (df["EventID"] == 2988) & (df["TrackID"] == 24)
df.loc[indy500_2014, "TrackID"] = 4                     # 2014 Indy 500 rows typed with another track's ID
ferrucci = (df["DriverName"] == "Santino Ferrucci") & (df["DriverID"] == 4401)
df.loc[ferrucci, "DriverID"] = 4897                     # Ferrucci's 2026 races typed with Katherine Legge's ID
print(f"Corrected {indy500_2014.sum()} Indy 500 TrackIDs and {ferrucci.sum()} Ferrucci DriverIDs")

check_results(df)

# Race folders are named by INDYCAR EventID, so the scraped CSVs carry it directly
for scraped in [pit_stops_df, race_metadata_df, cautions_df, lap_positions_df, lap_times_df]:
    ids = pd.to_numeric(scraped["event_id"], errors="coerce")
    if ids.isna().any():
        raise ValueError("Some scraped rows still use old folder names. "
                         "Run the one-time migration cell in indycar_pdf_scrapper first.")
    scraped["event_id"] = ids.astype(int)

valid_events = set(race_metadata_df["event_id"].unique())
pit_stops_df = pit_stops_df[pit_stops_df["event_id"].isin(valid_events)]
lap_positions_df = lap_positions_df[lap_positions_df["event_id"].isin(valid_events)]
lap_times_df = lap_times_df[lap_times_df["event_id"].isin(valid_events)]
cautions_df = cautions_df[cautions_df["event_id"].isin(valid_events)]

In [ ]:
pit_stops_df["DriverName"] = pit_stops_df["FName"] + " " + pit_stops_df["LName"]
pit_stops_df = pit_stops_df.drop(columns=["FName", "LName"])

lap_positions_df["DriverName"] = lap_positions_df["FName"] + " " + lap_positions_df["LName"]
lap_positions_df = lap_positions_df.drop(columns=["FName", "LName"])

lap_times_df["DriverName"] = lap_times_df["FName"] + " " + lap_times_df["LName"]
lap_times_df = lap_times_df.drop(columns=["FName", "LName"])

name_fixes = {
    "Patricio O'Ward": "Pato O'Ward",
    "Rinus Veekay": "Rinus VeeKay",
    "EJ Viso": "E.J. Viso",
    "Simona de Silvestro": "Simona De Silvestro",
}

pit_stops_df["DriverName"] = pit_stops_df["DriverName"].replace(name_fixes)
lap_positions_df["DriverName"] = lap_positions_df["DriverName"].replace(name_fixes)
lap_times_df["DriverName"] = lap_times_df["DriverName"].replace(name_fixes)

# The PDF data is joined to the results by driver name, so a spelling difference
# silently leaves that driver's pit and lap columns empty. List them instead.
def report_unmatched(scraped, label):
    known = set(zip(df["EventID"], df["DriverName"]))
    pairs = scraped[["event_id", "DriverName"]].drop_duplicates()
    unmatched = pairs[[(e, n) not in known for e, n in zip(pairs["event_id"], pairs["DriverName"])]]
    if len(unmatched):
        print(f"⚠️ {label}: {len(unmatched)} driver-race pairs match no one in the results. "
              "Add their spelling to name_fixes:")
        print(unmatched["DriverName"].value_counts().to_string())

report_unmatched(pit_stops_df, "Pit stops")
report_unmatched(lap_positions_df, "Lap chart")
report_unmatched(lap_times_df, "Lap times")

In [ ]:
pit_meta_df = pit_stops_df.merge(
    race_metadata_df[["event_id", "TotalRaceLaps"]], on="event_id", how="left"
)

In [ ]:
ritmo = 5
DNFwindow = 20
baseELO = 1500.0
k = 64
tk = 48
ek = 32

ordered_cols = [
    "DriverName", "DriverID", "Rookie",
    "DRFAvg", "DTAvg", "DTTAvg", "DNFRate", "TDNFRate",
    "DriverElo", "DriverTElo", "DriverTTElo", "DriverRitmo",
    "PositionStart", "StartVsField",
    "TeamName", "TeamID", "TRP", "TTP", "TeamDNFRate",
    "TeamElo", "TeamTElo", "TeamRitmo",
    "TeamTrackAvgPitStops", "TeamTrackAvgPittedCautionPCT",
    "CarEngine", "EngineID", "EngineElo", "EngineTElo", "EngineTTElo",
    "EventName", "Track", "TrackID",
    "EventTrackType", "EventTrackTypeID", "TrackAvgCautions", "TrackAvgCautionLaps",
    "TrackTypeAvgCautions", "TrackTypeAvgCautionLaps",  "TrackAvgSpeed",
    "EventDate", "EventDateFormatted", "EventID",
    "Era", "EraID",
    "Status", "StatusID", "FieldSize",
    "TotalCautions", "TotalCautionLaps", "TotalLeadChanges",
    "TotalRaceLaps", "TrackLength", "AvgRaceSpeed", "FastestLapSpeed",
    "AvgStintLength", "StintVariance",
    "FirstStopLap", "FirstStopVSAvgFirstStop",
    "PittedCautions", "PitStops", "PittedCautionPCT",
    "TotalPitStops", "PitStrategy",
    "BestRacePosition", "WorstRacePosition",
    "LapsLed", "Top5Laps", "Top10Laps", "PositionVolatility",
    "position_lap1", "PositionsGainedLap1",
    "PositionsGainedPits", "PositionsGainedCaution",
    "BestLapSpeed", "LapConsistency",
    "PaceDeg", "AvgPaceVSLeadPace",
    "AvgPitTime", "PitAvgTimeVSOverall",
    "TrackAvgPitStops", "FuelWindowEstimate",
    "PositionFinish", "NormalizedPositionFinish",
]

In [ ]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID"]).reset_index(drop=True)

df["NormalizedPositionFinish"] = (
    (df["PositionFinish"]-1)/(df["FieldSize"]-1)
)

In [ ]:
df["DRFAvg"] = (
    df.groupby("DriverID")["NormalizedPositionFinish"].transform(lambda s: s.shift(1).rolling(ritmo, min_periods=1).mean())
)

df["DTAvg"] = (
    df.groupby(["DriverID", "TrackID"])["NormalizedPositionFinish"].transform(lambda s: s.shift(1).expanding().mean())
)

df['DTTAvg'] = (
    df.groupby(["DriverID", "EventTrackTypeID"])["NormalizedPositionFinish"].transform(lambda s: s.shift(1).expanding().mean())
)

df["DNFRate"] = (
    df.groupby("DriverID")["StatusID"].transform((lambda s: s.shift(1).rolling(DNFwindow, min_periods=1).mean()))
)

df["TDNFRate"] = (
    df.groupby(["DriverID", "TrackID"])["StatusID"].transform((lambda s: s.shift(1).expanding().mean()))
)

In [ ]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID"]).reset_index(drop=True)

driverElo ={}
df["DriverElo"] = np.nan

for eventID, event in df.groupby("EventID", sort=False):
  indx = event.index
  drivers = event["DriverID"].values
  ranks = event["PositionFinish"].values.astype(float)

  preElo = np.array([driverElo.get(d, baseELO) for d in drivers], dtype=float)
  df.loc[indx, "DriverElo"] = preElo

  postElo = computeELO(preElo, ranks, k)
  for d, new in zip(drivers, postElo):
    driverElo[d] = float(new)


driverTrackElo = {}
df["DriverTElo"] = np.nan

for eventID, event in df.groupby("EventID", sort=False):
    indx = event.index
    drivers = event["DriverID"].values
    tracks = event["TrackID"].values
    ranks = event["PositionFinish"].values.astype(float)

    preElo = np.array([driverTrackElo.get((d, t), baseELO) for d, t in zip(drivers, tracks)], dtype=float)
    df.loc[indx, "DriverTElo"] = preElo

    postElo = computeELO(preElo, ranks, k)
    for d, t, new in zip(drivers, tracks, postElo):
        driverTrackElo[(d, t)] = float(new)

driverTrackTypeElo = {}
df["DriverTTElo"] = np.nan

for eventID, event in df.groupby("EventID", sort=False):
    indx = event.index
    drivers = event["DriverID"].values
    tracktypes = event["EventTrackTypeID"].values
    ranks = event["PositionFinish"].values.astype(float)

    preElo = np.array([driverTrackTypeElo.get((d, tt), baseELO) for d, tt in zip(drivers, tracktypes)], dtype=float)
    df.loc[indx, "DriverTTElo"] = preElo

    postElo = computeELO(preElo, ranks, k)
    for d, tt, new in zip(drivers, tracktypes, postElo):
        driverTrackTypeElo[(d, tt)] = float(new)

In [ ]:
df["DriverRitmo"] = (
    df.groupby("DriverID")["NormalizedPositionFinish"].transform(lambda s: s.shift(1).ewm(ritmo, min_periods=1).mean())
)

In [ ]:
df = df.drop(columns=["TRP", "TTP", "TeamDNFRate", "TeamElo",  "TeamTElo", "TeamRitmo"])
df["EventDate"] = pd.to_datetime(df["EventDate"])

teamdf = (
    df.groupby(["EventID", "TeamID"], as_index=False)
    .agg(
        teamPerf=("NormalizedPositionFinish", "mean"),
        EventTrackTypeID=("EventTrackTypeID", "first"),
        TrackID=("TrackID", "first"),
        EventDate=("EventDate", "first"),
        teamDNF=("StatusID", "max")
    ).sort_values(["EventDate", "EventID", "TeamID"]).reset_index(drop=True)
)

teamdf = teamdf.sort_values(["EventDate", "EventID", "TeamID"]).reset_index(drop=True)

In [ ]:
teamdf["TRP"] = (
    teamdf.groupby("TeamID")["teamPerf"].transform(lambda s: s.shift(1).rolling(ritmo, min_periods=1).mean())
)

teamdf["TTP"] = (
    teamdf.groupby(["TeamID", "EventTrackTypeID"])["teamPerf"].transform(lambda s: s.shift(1).expanding().mean())
)

teamdf["TeamDNFRate"] = (
     teamdf.groupby("TeamID")["teamDNF"].transform(lambda s: s.shift(1).rolling(DNFwindow, min_periods=1).mean())
)

In [ ]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID", "TeamID"]).reset_index(drop=True)

teamElo ={}
teamdf["TeamElo"] = np.nan

for eventID, event in teamdf.groupby("EventID", sort=False):
  indx = event.index
  teams = event["TeamID"].values
  ranks = event["teamPerf"].rank(method="first").values.astype(float)

  preElo = np.array([teamElo.get(t, baseELO) for t in teams], dtype=float)
  teamdf.loc[indx, "TeamElo"] = preElo

  postElo = computeELO(preElo, ranks, tk)
  for t, new in zip(teams, postElo):
    teamElo[t] = float(new)

teamTrackElo = {}
teamdf["TeamTElo"] = np.nan

for eventID, event in teamdf.groupby("EventID", sort=False):
    indx = event.index
    teams = event["TeamID"].values
    tracks = event["TrackID"].values
    ranks = event["teamPerf"].rank(method="first").values.astype(float)

    preElo = np.array([teamTrackElo.get((t, tr), baseELO) for t, tr in zip(teams, tracks)], dtype=float)
    teamdf.loc[indx, "TeamTElo"] = preElo

    postElo = computeELO(preElo, ranks, tk)
    for t, tr, new in zip(teams, tracks, postElo):
        teamTrackElo[(t, tr)] = float(new)

teamdf["TeamRitmo"] = (
    teamdf.groupby("TeamID")["teamPerf"].transform(lambda s: s.shift(1).ewm(ritmo, min_periods=1).mean())
)

In [ ]:
df = df.merge(
    teamdf[["EventID", "TeamID", "TRP", "TTP", "TeamDNFRate", "TeamElo", "TeamTElo", "TeamRitmo"]],
    on=["EventID", "TeamID"],
    how="left"
)

df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID", "TeamID", "DriverID"]).reset_index(drop=True)

In [ ]:
df = df.drop(columns=["EngineElo", "EngineTElo", "EngineTTElo"])
df["EventDate"] = pd.to_datetime(df["EventDate"])

enginedf = (
    df.groupby(["EventID", "EngineID"], as_index=False)
    .agg(
        enginePerf=("NormalizedPositionFinish", "mean"),
        EventTrackTypeID=("EventTrackTypeID", "first"),
        TrackID=("TrackID", "first"),
        EventDate=("EventDate", "first"),
    ).sort_values(["EventDate", "EventID", "EngineID"]).reset_index(drop=True)
)

In [ ]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID", "EngineID"]).reset_index(drop=True)

engineElo ={}
enginedf["EngineElo"] = np.nan

for eventID, event in enginedf.groupby("EventID", sort=False):
  indx = event.index
  engines = event["EngineID"].values
  ranks = event["enginePerf"].rank(method="first").values.astype(float)

  preElo = np.array([engineElo.get(e, baseELO) for e in engines], dtype=float)
  enginedf.loc[indx, "EngineElo"] = preElo

  postElo = computeELO(preElo, ranks, ek)
  for e, new in zip(engines, postElo):
    engineElo[e] = float(new)

engineTrackElo = {}
enginedf["EngineTElo"] = np.nan

for eventID, event in enginedf.groupby("EventID", sort=False):
    indx = event.index
    engines = event["EngineID"].values
    tracks = event["TrackID"].values
    ranks = event["enginePerf"].rank(method="first").values.astype(float)

    preElo = np.array([engineTrackElo.get((e, tr), baseELO) for e, tr in zip(engines, tracks)], dtype=float)
    enginedf.loc[indx, "EngineTElo"] = preElo

    postElo = computeELO(preElo, ranks, ek)
    for e, tr, new in zip(engines, tracks, postElo):
        engineTrackElo[(e, tr)] = float(new)

engineTrackTypeElo = {}
enginedf["EngineTTElo"] = np.nan

for eventID, event in enginedf.groupby("EventID", sort=False):
    indx = event.index
    engines = event["EngineID"].values
    tracktypes = event["EventTrackTypeID"].values
    ranks = event["enginePerf"].rank(method="first").values.astype(float)

    preElo = np.array([engineTrackTypeElo.get((e, tt), baseELO) for e, tt in zip(engines, tracktypes)], dtype=float)
    enginedf.loc[indx, "EngineTTElo"] = preElo

    postElo = computeELO(preElo, ranks, ek)
    for e, tt, new in zip(engines, tracktypes, postElo):
        engineTrackTypeElo[(e, tt)] = float(new)

In [ ]:
df = df.merge(
    enginedf[["EventID", "EngineID", "EngineElo", "EngineTElo", "EngineTTElo"]],
    on=["EventID", "EngineID"],
    how="left"
)

df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID", "EngineID", "DriverID"]).reset_index(drop=True)

In [ ]:
def compute_stint_metrics(group):
  group = group.sort_values("StopNumber")
  stop_laps = group["DriverLap"].values
  total_laps = group["TotalRaceLaps"].iloc[0]

  bounds = np.concatenate([[0], stop_laps, [total_laps]])
  stint_lengths = np.diff(bounds)

  return pd.Series({
      "TotalPitStops": group["TotalPitStops"].iloc[0],
      "TotalRaceLaps": total_laps,
      "AvgStintLength": stint_lengths.mean(),
      "StintVariance": stint_lengths.var(),
  })

stint_metrics_df = (
  pit_meta_df.groupby(["event_id", "DriverName", "carNum"])
  .apply(compute_stint_metrics, include_groups=False)
  .reset_index()
)

In [ ]:
def count_caution_pit(group):
  event = group.name[0]
  stop_laps = group.sort_values("StopNumber")["DriverLap"].values

  event_cautions = cautions_df[cautions_df["event_id"] == event]
  cautions_pit = 0
  for lap in stop_laps:
    is_under_caution = (
        (event_cautions["start_lap"] <= lap) &
        (lap <= event_cautions["end_lap"])
    ).any()
    if is_under_caution:
      cautions_pit += 1

  return pd.Series({
      "PittedCautions": cautions_pit,
      "PitStops": len(stop_laps),
      "PittedCautionPCT": cautions_pit / len(stop_laps) if len(stop_laps) > 0 else 0
  })

caution_pit_df = (
    pit_meta_df
    .groupby(["event_id", "DriverName", "carNum"])
    .apply(count_caution_pit, include_groups=False)
    .reset_index()
)

In [ ]:
def compute_position_summary(group):
    positions = group.sort_values("lap")["position"]

    return pd.Series({
        "BestRacePosition": positions.min(),
        'WorstRacePosition': positions.max(),
        "LapsLed": (positions == 1).sum(),
        "Top5Laps": (positions <= 5).sum(),
        "Top10Laps": (positions <= 10).sum(),
        "PositionVolatility": positions.std(),
    })

position_summary = (
    lap_positions_df
    .groupby(["event_id", "DriverName","car"])
    .apply(compute_position_summary, include_groups=False)
    .reset_index()
)

In [ ]:
def compute_pit_cycle_gains(group):
    event = group.name[0]
    car = group.name[2]

    stops = pit_meta_df[
        (pit_meta_df["event_id"] == event) &
        (pit_meta_df["carNum"] == car)
    ].sort_values("StopNumber")

    if len(stops) == 0:
        return pd.Series({"PositionsGainedPits": 0})

    total_gain = 0
    for _, stop in stops.iterrows():
        stop_lap = stop["DriverLap"]

        before = group[group["lap"] == stop_lap - 1]["position"]
        after = group[group["lap"] == stop_lap + 10]["position"]

        if len(before) > 0 and len(after) > 0:
            total_gain += before.iloc[0] - after.iloc[0]

    return pd.Series({"PositionsGainedPits": total_gain})

pit_cycle_gains = (
    lap_positions_df
    .groupby(["event_id", "DriverName", "car"])
    .apply(compute_pit_cycle_gains, include_groups=False)
    .reset_index()
)

In [ ]:
def compute_caution_gains(group):
    event = group.name[0]

    event_cautions = cautions_df[cautions_df["event_id"] == event]

    if len(event_cautions) == 0:
        return pd.Series({"PositionsGainedCaution": 0})

    total_gain = 0
    for _, caution in event_cautions.iterrows():
        start = caution["start_lap"]
        end = caution["end_lap"]

        before = group[group["lap"] == start - 1]["position"]
        after = group[group["lap"] == end + 1]["position"]

        if len(before) > 0 and len(after) > 0:
            total_gain += before.iloc[0] - after.iloc[0]

    return pd.Series({"PositionsGainedCaution": total_gain})

caution_gains = (
    lap_positions_df
    .groupby(["event_id", "DriverName", "car"])
    .apply(compute_caution_gains, include_groups=False)
    .reset_index()
)

In [ ]:
def compute_pace_summary(group):
  event = group.name[0]
  car = group.name[2]

  group = group.sort_values("lap")

  pit_stops = pit_meta_df[
      (pit_meta_df["event_id"] == event) &
      (pit_meta_df["carNum"] == car)
  ]["DriverLap"].values

  race_cautions = cautions_df[cautions_df["event_id"] == event]
  caution_laps = set()
  for _, c in race_cautions.iterrows():
    for lap in range(c["start_lap"], c["end_lap"] + 1):
      caution_laps.add(lap)

  green = group[
      (~group["lap"].isin(pit_stops)) &
      (~group["lap"].isin(caution_laps))]["lap_time"]

  return pd.Series({
      "BestLapSpeed": group["lap_time"].min(),
      "LapConsistency": green.std() if len(green) > 0 else np.nan
  })

pace_summary = (
    lap_times_df
    .groupby(["event_id", "DriverName", "car"])
    .apply(compute_pace_summary, include_groups=False)
  .reset_index()
)

In [ ]:
def compute_pace_degradation(group):
    event = group.name[0]
    car = group.name[2]

    group = group.sort_values("lap")

    pit_stops = pit_meta_df[
        (pit_meta_df["event_id"] == event) &
        (pit_meta_df["carNum"] == car)
    ].sort_values("StopNumber")["DriverLap"].values

    total_laps = race_metadata_df[race_metadata_df["event_id"] == event]["TotalRaceLaps"].iloc[0]
    boundaries = np.concatenate([[1], pit_stops, [total_laps]])

    race_cautions = cautions_df[cautions_df["event_id"] == event]
    caution_laps = set()
    for _, c in race_cautions.iterrows():
        for lap in range(c["start_lap"], c["end_lap"] + 1):
            caution_laps.add(lap)

    slopes = []
    for i in range(len(boundaries) - 1):
        stint_start = boundaries[i]
        stint_end = boundaries[i + 1]

        stint = group[
            (group["lap"] >= stint_start) &
            (group["lap"] < stint_end) &
            (~group["lap"].isin(pit_stops)) &
            (~group["lap"].isin(caution_laps))
        ]

        if len(stint) >= 3:
            coeffs = polyfit(stint["lap"].values, stint["lap_time"].values, 1)
            slopes.append(coeffs[1])

    return pd.Series({
        "PaceDeg": np.mean(slopes) if len(slopes) > 0 else np.nan
    })

pace_degradation = (
    lap_times_df
    .groupby(["event_id", "DriverName", "car"])
    .apply(compute_pace_degradation, include_groups=False)
    .reset_index()
)

In [ ]:
# 10 min
def compute_pace_vs_leader(group):
    event = group.name[0]
    car = group.name[2]

    group = group.sort_values("lap")

    race_laps = lap_times_df[lap_times_df["event_id"] == event]
    leader_pace = race_laps.groupby("lap")["lap_time"].min()

    race_cautions = cautions_df[cautions_df["event_id"] == event]
    caution_laps = set()
    for _, c in race_cautions.iterrows():
        for lap in range(c["start_lap"], c["end_lap"] + 1):
            caution_laps.add(lap)

    pit_stops = pit_meta_df[
        (pit_meta_df["event_id"] == event) &
        (pit_meta_df["carNum"] == car)
    ]["DriverLap"].values


    green = group[
        (~group['lap'].isin(pit_stops)) &
        (~group['lap'].isin(caution_laps))
    ]

    gaps = []
    for _, row in green.iterrows():
        lap_num = row["lap"]
        if lap_num in leader_pace.index and lap_num not in caution_laps:
            gap = row["lap_time"] - leader_pace[lap_num]
            gaps.append(gap)

    return pd.Series({
        "AvgPaceVSLeadPace": np.mean(gaps) if len(gaps) > 0 else np.nan
    })

pace_vs_leader = (
    lap_times_df
    .groupby(["event_id", "DriverName", "car"])
    .apply(compute_pace_vs_leader, include_groups=False)
    .reset_index()
)

In [ ]:
def compute_pit_duration(group):
    event = group.name[0]
    car = group.name[2]

    group = group.sort_values("lap")

    pit_stops = pit_meta_df[
        (pit_meta_df["event_id"] == event) &
        (pit_meta_df["carNum"] == car)
    ]["DriverLap"].values

    race_cautions = cautions_df[cautions_df["event_id"] == event]
    caution_laps = set()
    for _, c in race_cautions.iterrows():
        for lap in range(c["start_lap"], c["end_lap"] + 1):
            caution_laps.add(lap)

    green = group[
        (~group["lap"].isin(pit_stops)) &
        (~group["lap"].isin(caution_laps))
    ]["lap_time"]

    normal_time = green.median()

    pit_lap_times = group[group["lap"].isin(pit_stops)]["lap_time"]
    pit_durations = pit_lap_times - normal_time

    return pd.Series({
        "AvgPitTime": pit_durations.mean() if len(pit_durations) > 0 else np.nan,
        "PitAvgTimeVSOverall": np.nan  # placeholder, computed after
    })

pit_duration = (
    lap_times_df
    .groupby(["event_id", "DriverName", "car"])
    .apply(compute_pit_duration, include_groups=False)
    .reset_index()
)

field_avg_pit = (
    pit_duration
    .groupby("event_id")["AvgPitTime"]
    .transform("mean")
)

pit_duration["PitAvgTimeVSOverall"] = pit_duration["AvgPitTime"] - field_avg_pit

In [ ]:
pit_strategy_map = {1: '1-stop', 2: '2-stop', 3: '3-stop', 4: '4-stop', 5: '5-stop'}

strategy_type_df = (
    pit_meta_df
    .groupby(["event_id", "DriverName", "carNum"])["TotalPitStops"]
    .first()
    .reset_index()
)

strategy_type_df["PitStrategy"] = strategy_type_df["TotalPitStops"].map(pit_strategy_map).fillna("Other")

first_stop_df = (
    pit_meta_df
    .sort_values("StopNumber")
    .groupby(["event_id", "DriverName", "carNum"])
    .first()
    .reset_index()
    [["event_id", "DriverName", "carNum", "DriverLap"]]
    .rename(columns={"DriverLap": "FirstStopLap"})
)

field_avg_first_stop_df = (
    first_stop_df
    .groupby("event_id")["FirstStopLap"]
    .transform("mean")
)

first_stop_df["FirstStopVSAvgFirstStop"] = first_stop_df["FirstStopLap"] - field_avg_first_stop_df

In [ ]:
lap1_positions = lap_positions_df[lap_positions_df['lap'] == 1][['event_id', 'car', 'DriverName', 'position']]
lap1_positions = lap1_positions.rename(columns={'position': 'position_lap1'})

In [ ]:
race_metadata_df.rename(columns={"event_id": "EventID"}, inplace=True)

for frame in [race_metadata_df, stint_metrics_df, first_stop_df, caution_pit_df,
              strategy_type_df, position_summary, lap1_positions, pit_cycle_gains,
              caution_gains, pace_summary, pace_degradation, pace_vs_leader, pit_duration]:
    frame.rename(columns={"event_id": "EventID"}, inplace=True)

In [ ]:
# Drop the copies these merges would duplicate (as with the team and engine columns above),
# so the input can be any earlier output, e.g. v25 with a new race typed in
scraped_columns = [
    "TotalCautions", "TotalCautionLaps", "TotalLeadChanges", "TotalRaceLaps", "TrackLength",
    "AvgRaceSpeed", "FastestLapSpeed", "AvgStintLength", "StintVariance", "FirstStopLap",
    "FirstStopVSAvgFirstStop", "PittedCautions", "PitStops", "PittedCautionPCT", "TotalPitStops",
    "PitStrategy", "BestRacePosition", "WorstRacePosition", "LapsLed", "Top5Laps", "Top10Laps",
    "PositionVolatility", "position_lap1", "PositionsGainedPits", "PositionsGainedCaution",
    "BestLapSpeed", "LapConsistency", "PaceDeg", "AvgPaceVSLeadPace", "AvgPitTime", "PitAvgTimeVSOverall",
]
df = df.drop(columns=[c for c in scraped_columns if c in df.columns])

# Race metadata — one row per race, every driver gets the same values
df = df.merge(
    race_metadata_df[["EventID", "TotalCautions", "TotalCautionLaps",
                       "TotalLeadChanges", "TotalRaceLaps", "TrackLength",
                       "AvgRaceSpeed", "FastestLapSpeed"]],
    on="EventID",
    how="left"
)

# Stint metrics (avg_stint_length, stint_length_variance)
df = df.merge(
    stint_metrics_df[["EventID", "DriverName", "AvgStintLength", "StintVariance"]],
    on=["EventID", "DriverName"],
    how="left"
)

# First stop + first stop vs field avg
df = df.merge(
    first_stop_df[["EventID", "DriverName", "FirstStopLap", "FirstStopVSAvgFirstStop"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Caution pit metrics
df = df.merge(
    caution_pit_df[["EventID", "DriverName", "PittedCautions", "PitStops", "PittedCautionPCT"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Strategy type
df = df.merge(
    strategy_type_df[["EventID", "DriverName", "TotalPitStops", "PitStrategy"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Position summary (best, worst, laps led, top5, top10, volatility)
df = df.merge(
    position_summary[["EventID", "DriverName", "BestRacePosition", "WorstRacePosition",
                       "LapsLed", "Top5Laps", "Top10Laps", "PositionVolatility"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Lap 1 positions
df = df.merge(
    lap1_positions[["EventID", "DriverName", "position_lap1"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Positions gained on pit cycles
df = df.merge(
    pit_cycle_gains[["EventID", "DriverName", "PositionsGainedPits"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Positions gained under caution
df = df.merge(
    caution_gains[["EventID", "DriverName", "PositionsGainedCaution"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Pace summary (fastest lap, consistency)
df = df.merge(
    pace_summary[["EventID", "DriverName", "BestLapSpeed", "LapConsistency"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Pace degradation
df = df.merge(
    pace_degradation[["EventID", "DriverName", "PaceDeg"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Pace vs leader
df = df.merge(
    pace_vs_leader[["EventID", "DriverName", "AvgPaceVSLeadPace"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Pit duration
df = df.merge(
    pit_duration[["EventID", "DriverName", "AvgPitTime", "PitAvgTimeVSOverall"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Positions gained lap 1 (needs PositionStart which is already in df)
df["PositionsGainedLap1"] = df["PositionStart"] - df["position_lap1"]

df["StartVsField"] = df["PositionStart"] / df["FieldSize"]

In [ ]:
df = df.sort_values(["EventDate", "EventID", "DriverID"]).reset_index(drop=True)

def prior_race_mean(df, keys, value, once_per_race):
    """Average of `value` over earlier races with the same `keys`, for every row.

    A race's own values join the history only after all of its rows are filled in,
    so no driver sees the race being predicted. once_per_race=True for race-level
    values (cautions, speed) that repeat on every car's row.
    """
    history = {}
    result = pd.Series(np.nan, index=df.index)
    for _, race in df.groupby("EventID", sort=False):
        race_keys = list(race[keys].itertuples(index=False, name=None))
        result.loc[race.index] = [np.mean(history[k]) if history.get(k) else np.nan for k in race_keys]
        added = race.drop_duplicates(keys) if once_per_race else race
        for k, v in zip(added[keys].itertuples(index=False, name=None), added[value]):
            if pd.notna(v):
                history.setdefault(k, []).append(v)
    return result

df["TrackAvgPitStops"] = prior_race_mean(df, ["TrackID"], "TotalPitStops", once_per_race=False)
df["FuelWindowEstimate"] = df["TotalRaceLaps"] / df["TrackAvgPitStops"]
df["TrackAvgCautions"] = prior_race_mean(df, ["TrackID"], "TotalCautions", once_per_race=True)
df["TrackAvgCautionLaps"] = prior_race_mean(df, ["TrackID"], "TotalCautionLaps", once_per_race=True)
df["TrackTypeAvgCautions"] = prior_race_mean(df, ["EventTrackTypeID"], "TotalCautions", once_per_race=True)
df["TrackTypeAvgCautionLaps"] = prior_race_mean(df, ["EventTrackTypeID"], "TotalCautionLaps", once_per_race=True)
df["TrackAvgSpeed"] = prior_race_mean(df, ["TrackID"], "AvgRaceSpeed", once_per_race=True)
df["TeamTrackAvgPitStops"] = prior_race_mean(df, ["TeamID", "TrackID"], "TotalPitStops", once_per_race=False)
df["TeamTrackAvgPittedCautionPCT"] = prior_race_mean(df, ["TeamID", "TrackID"], "PittedCautionPCT", once_per_race=False)

In [ ]:
df = df.sort_values(["EventDate", "EventID", "DriverID"]).reset_index(drop=True)
df = df[ordered_cols]

df.to_csv(BASE_DIR / OUTPUT_FILE, index=False)